In [63]:
# Imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

## Load Cluster Node Profiles

In [35]:
def load_node_profile(node):
    filepath = f'../../tool/infrastructure-profiler/profiles/{node}.csv'

    with open(filepath) as f:
        data = f.readlines()[1].split(',')

    return {
        'ram': int(data[1].strip()),
        'cores': int(data[2].strip()),
        'io_read': float(data[3].strip()),
        'io_write': float(data[4].strip()),
        'sysbench': float(data[5].strip()),
        'z7b': float(data[6].strip()),
        'bench_time': float(data[7].strip())
    }

In [36]:
nodes = ['aws-metal-c5n', 'aws-metal-z1d', 'gpgnode-16', 'gpgnode-22', 'tu-06', 'hu-c40', 'hu-c42']
node_profiles = {}

for node in nodes:
    profile = load_node_profile(node)
    node_profiles[node] = profile

In [37]:
def combine_df(input_df, trace_df):
    rows = []
    assert len(trace_df) == len(input_df['name'].unique())

    for name, group in input_df.groupby('name'):
        no_files = str(len(group))
        filepaths = group['path']
        paths = ','.join([filepath.split('/')[-1].split('.')[-1] for filepath in filepaths])
        total_size = group['size'].sum()  # / 1073741824  # B to GiB
        task_record = trace_df[trace_df['name'] == name].iloc[0]
        row = [task_record['task_id'], task_record['name'], task_record['%cpu'], 
               task_record['start'], task_record['complete'], task_record['complete'] - task_record['start'], 
               no_files, paths, total_size, task_record['name'].split(':')[-1].split(' ')[0]]
        rows.append(row)

    combined = pd.DataFrame(rows, columns=['id', 'name', '%cpu', 'start', 'complete', 'runtime', 'no_files', 'files', 'input_size_b', 'short_name'])
    return combined

In [38]:
def load_profiling_runs(node, workflows, samples):
    node_profile = node_profiles[node]
    runs = {}

    for workflow, samples in zip(workflows, samples):
        runs[workflow] = {}

        for sample in samples:
            sample_dir = f'{node}/{workflow}/{sample}/'
            input_df = pd.read_csv(sample_dir + 'input.csv', sep=';')
            trace_df = pd.read_csv(sample_dir + 'trace.csv')
            runs[workflow][sample] = combine_df(input_df, trace_df)

    return (node_profile, runs)

In [39]:
def load_workflow_execution(cluster, workflow, iteration):
    execution_dir = f'../{cluster}/{workflow}/{iteration}/'
    input_df = pd.read_csv(execution_dir + 'input.csv', sep=';')
    trace_df = pd.read_csv(execution_dir + 'trace.csv')
    return combine_df(input_df, trace_df)

## Predict from TU-06

In [40]:
TU_06 = 'tu-06'
TU_06_workflows = ['rnaseq', 'chipseq']
TU_06_samples = [['sample_b', 'sample_c', 'sample_d'], ['sample_e', 'sample_f', 'sample_g', 'sample_h']]
(TU_06_profile, TU_06_runs) = load_profiling_runs(TU_06, TU_06_workflows, TU_06_samples)

In [41]:
HU_40 = 'hu-c40'
HU_40_workflows = ['rnaseq', 'chipseq']
HU_40_samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d'], ['sample_e', 'sample_f', 'sample_g', 'sample_h']]
(HU_40_profile, HU_40_runs) = load_profiling_runs(HU_40, HU_40_workflows, HU_40_samples)

In [42]:
HU_42 = 'hu-c42'
HU_42_workflows = ['rnaseq', 'chipseq']
HU_42_samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d'], ['sample_e', 'sample_f', 'sample_g', 'sample_h']]
(HU_42_profile, HU_42_runs) = load_profiling_runs(HU_42, HU_42_workflows, HU_42_samples)

In [43]:
# identify real executions on hu-cluster
hu_rnaseq = load_workflow_execution('hu-cluster', 'rnaseq', 1)
hu_chipseq = load_workflow_execution('hu-cluster', 'chipseq', 1)

In [44]:
# hu-c40 -- rnaseq sample_d profiling 
comb_rnaseq_sample_d = HU_40_runs['rnaseq']['sample_d']

# q: how to use multiple runs i.e. diff samples to create this training dataset?
linear_regression = LinearRegression()
linear_regression.fit(comb_rnaseq_sample_d[['input_size_b', 'no_files']], comb_rnaseq_sample_d['runtime'].values)

# use model to predict duration based on input size from a FULL WORKFLOW RUN ON
# - the same cluster node type
# - a different cluster node type

for _, record in hu_rnaseq.iterrows():
    if record['input_size_b'] > 1696922320:
        print(record['input_size_b'], record['runtime'], record['no_files'], record['files'])
        break

# Example of how to use the model to predict duration for a new input size
new_input_size = 1696922320  # value in Bytes
predicted_duration = linear_regression.predict([[new_input_size, 2]])
print('Linear Regression')
print(f'input size {new_input_size}B | duration {predicted_duration[0]:.0f} ms')


1696927331 14000 2 bai,bam
Linear Regression
input size 1696922320B | duration 62989 ms


/usr/lib/python3/dist-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [54]:
def get_task_data_from_experiment(task_name, experiment):
    task_data = experiment[experiment['short_name'] == task_name]
    key_data = task_data[['input_size_b', 'runtime']]
    key_data.columns = ['x', 'y']
    key_data.reset_index().drop('index', axis=1)
    return key_data

In [ ]:
# train models for all tasks in rnaseq from profiling runs
tasks = HU_40_runs['rnaseq']['sample_a']['short_name'].unique()
task_models = {}

for task in tasks:
    task_data = pd.DataFrame(columns=['x', 'y'], dtype='float64')
    for sample in ['sample_a', 'sample_b', 'sample_c', 'sample_d']:
        experiment_data = get_task_data_from_experiment(task, HU_40_runs['rnaseq'][sample])
        task_data = pd.concat([task_data, experiment_data])

    lr = LinearRegression()
    task_models[task] = lr.fit(task_data['x'].values.reshape(-1, 1), task_data['y'].values) 

In [70]:
# make prediction using trained models
overall_error = []
missing_tasks = []

for _, record in hu_rnaseq.iterrows():
    task_name = record['short_name']
    data_size = record['input_size_b']
    actual_runtime = record['runtime']

    if task_name in task_models.keys():
        predicted_runtime = task_models[task_name].predict([[data_size]])
        

        if actual_runtime > 0:
            error = (abs(predicted_runtime - actual_runtime) / actual_runtime) * 100 
            overall_error.append(error)
        else:
            print(task_name, data_size, predicted_runtime, actual_runtime, error)

        print(f'{task_name} with data {data_size}B --> pred {predicted_runtime}ms actu {actual_runtime}ms erro {error}%')
    else:
        missing_tasks.append(task_name)

# print(f'avg. error {np.mean(overall_error)}')

SAMTOOLS_FLAGSTAT with data 1696922320B --> pred [14945.46535507]ms actu 14000ms erro [6.75332396]%
SAMTOOLS_FLAGSTAT with data 1696927331B --> pred [14945.50947078]ms actu 14000ms erro [6.75363908]%
SAMTOOLS_FLAGSTAT with data 1696892792B --> pred [14945.20539723]ms actu 16000ms erro [6.59246627]%
SAMTOOLS_FLAGSTAT with data 1767700102B --> pred [15568.57696445]ms actu 14000ms erro [11.20412117]%
SAMTOOLS_FLAGSTAT with data 1204529753B --> pred [10610.55238944]ms actu 10000ms erro [6.10552389]%
SAMTOOLS_FLAGSTAT with data 1493507927B --> pred [13154.65097502]ms actu 13000ms erro [1.18962288]%
SAMTOOLS_FLAGSTAT with data 2022648072B --> pred [17813.08131887]ms actu 17000ms erro [4.78283129]%
SAMTOOLS_FLAGSTAT with data 1828709600B --> pred [16105.69080741]ms actu 14000ms erro [15.04064862]%
SAMTOOLS_FLAGSTAT with data 1728736218B --> pred [15225.54772871]ms actu 17000ms erro [10.43795454]%
SAMTOOLS_FLAGSTAT with data 2101335401B --> pred [18505.82679387]ms actu 21000ms erro [11.8770152

In [ ]:
# Make a simple svm, linear regression and random forest regression model to predict y from x
# models = {
#     'SVR': SVR(),
#     'RandomForest': RandomForestRegressor(),
#     'LinearRegression': LinearRegression()
# }

# mse = mean_squared_error(y_test, y_pred) 
# print(f'Linear Regression MSE: {mse}')
